# 240 - k-means at every K, both feature sets

**Run 240, 241 and 242 in parallel** - three kernels, one per algorithm. They share the dataset cache read-only and write to separate run directories and separate sweep files, so they cannot collide.

**Before this:** `python rebuild_concat_cache.py --apply` must have finished. It is the only serial step.

**After all three:** run `249_cluster_statistics.ipynb` for the comparison and the figures. `243_cluster_archetypes.ipynb` is an optional fourth track and does not gate it.

| | |
|---|---|
| method | k-means |
| feature sets | `concat_hg`, `concat_rawds` (gated cohort, one set of electrodes) |
| K | 5 .. 30 |
| cNMF fit iterations | 1000 final, 300 inside cross-validation |


In [1]:
import os, sys, json, subprocess, time
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
assert (ROOT / 'functions').exists(), f'run this from 02_FBM_Clustering, not {ROOT}'
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / 'functions'))
# cfg is lf_blob_clustering_config here - 02_FBM_Clustering has no config.py;
# that name belongs to 01_FBM_Analysis. RANDOM_STATE lives in this one.
import lf_concat as CC, lf_cluster_run as R, lf_runs as LR
import lf_blob_clustering_config as cfg

SCRIPT_NAME = '240_cluster_kmeans.ipynb'
METHOD      = 'kmeans'
METHOD_LBL  = 'k-means'
CACHE_DIR   = ROOT / 'outputs/_dataset/concat_source_v4'
FEATURE_SETS= ['concat_hg', 'concat_rawds', 'concat_bands5', 'concat_bands5z']
K_RANGE     = [5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30]
N_ITER      = 1000          # convex NMF fit iterations (final fits)
RANDOM_STATE= cfg.RANDOM_STATE

INPUT_DIR = Path(r'\\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\01_FBM_Analysis\outputs\04_ersp_LM_RAWONLY')
if not INPUT_DIR.exists():
    INPUT_DIR = (ROOT.parent / '01_FBM_Analysis' / 'outputs' / '04_ersp_LM_RAWONLY').resolve()

print('method      :', METHOD_LBL)
print('feature sets:', FEATURE_SETS)
print('K range     :', K_RANGE[0], '..', K_RANGE[-1], f'({len(K_RANGE)} values)')
print('cache       :', CACHE_DIR)

method      : k-means
feature sets: ['concat_hg', 'concat_rawds', 'concat_bands5', 'concat_bands5z']
K range     : 5 .. 30 (26 values)
cache       : \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\02_FBM_Clustering\outputs\_dataset\concat_source_v4


## 1 - Load the shared cohort

Built once by `rebuild_concat_cache.py`. This notebook only reads it.

In [2]:
# The cache MUST already exist. rebuild_concat_cache.py builds it once, serially.
# Building it here would mean three kernels racing to write the same files, and
# prepare_dataset's cache key carries no patient list, so a half-written cache would
# be picked up as a hit by whichever kernel arrived second.
assert CACHE_DIR.exists() and (CACHE_DIR / 'X_3d.npy').exists(), (
    f'{CACHE_DIR} missing - run:  python rebuild_concat_cache.py --apply')

df_contacts, X_concat = CC.build_concat_dataset(
    INPUT_DIR, conditions=('audio','picture','reading'),
    require_high_activity=True, cache_dir=CACHE_DIR, verbose=True)

X = {'concat_hg':     CC.concat_hg_features(X_concat, hg_band=(70.0,150.0), fmax=500.0),
     'concat_rawds':  CC.concat_rawds_features(X_concat, n_blocks=3, fmax_hz=500.0),
     # concat_bands5 is the SAME builder with 5 band edges instead of 15, each a
     # union of contiguous 15-band edges. It averages the ORIGINAL frequency bins,
     # so a wide band is weighted by the bins it contains rather than giving a 3 Hz
     # sub-band the same say as a 40 Hz one - and the two sets stay nested.
     'concat_bands5': CC.concat_bands5_features(X_concat, n_blocks=3, fmax_hz=500.0),
     # the same five bands with each one z-scored to equal weight. NOT dB any
     # more - the units are SD within a band - and it is a COHORT-level rescale,
     # so it changes with the cohort and must be rebuilt when the cohort does.
     'concat_bands5z': CC.concat_bands5z_features(X_concat, n_blocks=3, fmax_hz=500.0)}

print(f'\ncohort: {len(df_contacts)} electrodes · '
      f'{df_contacts.patient_id.nunique()} patients')
for k_, v in X.items():
    print(f'  {k_:<14} {v.shape}')

[lf_dataset cache miss] params differ — rebuilding
[lf_dataset] detected 29 patient folders under \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\01_FBM_Analysis\outputs\04_ersp_LM_RAWONLY
  loaded 9774 samples
  excluded 15 non-neural channels → 9759 samples
  excluded 318 microelectrode channels (ADM, AGM, FODM, FOM, HADM, HAGM, IDM, PHDM, POM, TM, TPDM) → 9441 samples
  excluded 99 noise-contaminated channels {'PAT_3415': ['IMG', 'IPG', 'TA']} → 9342 samples
[lf_dataset] canonical dataset ready: 9342 samples · X_3d.shape=(9342, 129, 300)
  cached to \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\02_FBM_Clustering\outputs\_dataset\concat_source_v4
[lf_concat] dropped 192 subdural GRID contacts {'PAT_3415': 192} — their depth contacts are kept
[lf_concat] excluded patients ['EL044'] — 124 rows removed
[lf_concat] 1693 electrodes (audio+picture+reading) · X_concat=(1693, 129, 900)
[lf_concat]   dropped 149 (missing a condition), 1266 (no high-

## 2 - Fit k-means at every K

In [3]:
# one loop, every feature set, every K in one fit_and_save call
#
# REFITTING A FEATURE SET DOES NOT UPDATE ITS RUN - IT SUPERSEDES IT. Run ids are
# timestamps and lf_runs.newest_run takes the last, so a second fit of concat_hg
# leaves the first one's statistics, stability sweeps and coverage-manifest entry
# stranded on a run nothing resolves any more. Set SKIP_IF_RUN_EXISTS = False only
# when you actually mean to replace one.
SKIP_IF_RUN_EXISTS = True

results, skipped = {}, []
for fs in FEATURE_SETS:
    if SKIP_IF_RUN_EXISTS:
        try:
            rd_ = LR.newest_run(METHOD, fs)
            lab_ = pd.read_csv(rd_ / 'cluster_labels_by_k.csv')
            have_ = {int(c[2:]) for c in lab_.columns if c.startswith('k_')}
            if set(K_RANGE).issubset(have_):
                skipped.append(fs)
                print(f'{fs:<15} SKIPPED - {rd_.name} already covers this K range')
                continue
            print(f'{fs:<15} refitting - {rd_.name} is missing '
                  f'K={sorted(set(K_RANGE) - have_)}')
        except FileNotFoundError:
            pass
    t0 = time.time()
    params = ({'k_range': K_RANGE, 'random_state': RANDOM_STATE, 'n_init': 20}
              if METHOD == 'kmeans' else
              {'linkage': 'ward', 'metric': 'euclidean', 'k_range': K_RANGE})
    m = R.fit_and_save(
        X[fs], df_keep=df_contacts, method=METHOD, feature_set=fs, params=params,
        method_label=METHOD_LBL,
        # .get, not [fs]: a bare lookup makes a new feature set a KeyError
        # AFTER the features are built and the loop has started.
        feature_set_label={'concat_hg':     'Concatenated HG [a|p|r]',
                           'concat_rawds':  'Concatenated 15-band [a|p|r]',
                           'concat_bands5': 'Concatenated 5-band [a|p|r]',
                           'concat_bands5z':'Concatenated 5-band, z-scored [a|p|r]'}.get(fs, fs),
        feature_names=CC.concat_feature_names(fs), notebook=SCRIPT_NAME)
    results[fs] = m
    print(f'{fs:<15} best_k={m["summary"]["best_k"]}   ({time.time()-t0:.0f}s)')

if skipped:
    print(f'\n{len(skipped)} already fitted and left alone: {skipped}')

# verify EVERY feature set, skipped ones included - a skip is only safe if the run it
# skipped for really does hold the K range
for fs in FEATURE_SETS:
    rd = LR.newest_run(METHOD, fs)
    lab = pd.read_csv(rd / 'cluster_labels_by_k.csv')
    have = sorted(int(c[2:]) for c in lab.columns if c.startswith('k_'))
    tag = '(skipped)' if fs in skipped else ''
    print(f'{fs:<15} {rd.name}   K columns {have[0]}..{have[-1]} ({len(have)}) {tag}')
    assert set(K_RANGE).issubset(have), f'{fs}: missing K columns'

concat_hg       SKIPPED - 20260826_144334 already covers this K range
concat_rawds    SKIPPED - 20260826_144504 already covers this K range
  K=  5  sil=0.1005
  K=  6  sil=0.0796
  K=  7  sil=0.0831
  K=  8  sil=0.0772
  K=  9  sil=0.0679
  K= 10  sil=0.0738
  K= 11  sil=0.0829
  K= 12  sil=0.0806
  K= 13  sil=0.0758
  K= 14  sil=0.0858
  K= 15  sil=0.0928
  K= 16  sil=0.0889
  K= 17  sil=0.0925
  K= 18  sil=0.0838
  K= 19  sil=0.0889
  K= 20  sil=0.0906
  K= 21  sil=0.0915
  K= 22  sil=0.0895
  K= 23  sil=0.0918
  K= 24  sil=0.0918
  K= 25  sil=0.0920
  K= 26  sil=0.0964
  K= 27  sil=0.1045
  K= 28  sil=0.1082
  K= 29  sil=0.1008
  K= 30  sil=0.1003

[KMeans] Best K=28  silhouette=0.1082
[fit_and_save] sweep statistics for K=[5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30] -> sweep_metrics.csv
[fit_and_save] kmeans/concat_bands5/20260827_021650
  n_samples=1693 n_clusters=28 silhouette=0.108
  -> \\nasac-m2.unige.ch\m-HumanNeuronLab\

## 3 - Held-out variance, K=5..30

Bi-cross-validation: a block of rows AND a block of columns is held out, so an extra component has to earn its place. Home space only - that is what the peak-K decision reads.

In [5]:
# Held-out variance, K=5..30, HOME SPACE only, bi-cross-validated.
# --tag keeps this kernel's output in its own file so the three notebooks running in
# parallel never write to the same CSV.
cmd = [sys.executable, 'make_heldout_variance.py',
       '--from-cache', str(CACHE_DIR),
       '--feature-set', *FEATURE_SETS,
       '--method', METHOD, '--tag', METHOD,
       '--ks', *[str(k) for k in K_RANGE],
       '--spaces', 'home', '--n-iter', '300']
print('$', ' '.join(cmd), flush=True)
r = subprocess.run(cmd, cwd=str(ROOT), env={**os.environ, 'PYTHONIOENCODING':'utf-8'})
assert r.returncode == 0

pk = pd.read_csv(ROOT/'outputs'/'clustering'/'bsf_comparison'/f'heldout_peaks_{METHOD}.csv')
display(pk)

$ C:\Users\artoni\.conda\envs\LORA_ENV2\python.exe make_heldout_variance.py --from-cache \\nasac-m2.unige.ch\m-HumanNeuronLab\ANALYSIS\FLM\Analysis_LoraFanda\02_FBM_Clustering\outputs\_dataset\concat_source_v4 --feature-set concat_hg concat_rawds concat_bands5 concat_bands5z --method kmeans --tag kmeans --ks 5 6 7 8 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 --spaces home --n-iter 300


,feature_set,scheme,method_label,k_peak,peak,at_k8,monotone,k_max_tested
0,concat_bands5,home,k-means,30,0.521114,0.388173,False,30
1,concat_bands5z,home,k-means,30,0.533558,0.385965,False,30
2,concat_hg,home,k-means,30,0.588969,0.464581,True,30
3,concat_rawds,home,k-means,30,0.440238,0.324325,False,30


## 4 - Done

In [6]:
print('DONE:', METHOD_LBL)
print('When ALL THREE of 240 / 241 / 242 have finished, run 249.')

DONE: k-means
When ALL THREE of 240 / 241 / 242 have finished, run 249.
